# PSELDNets × v10.2（規模拡大3828本＋複数車対応追加705本、日本適合・消防車・バック混合込み）

事前準備（ローカル→Drive `MyDrive/PSELDNets_data/`）:
- `dataset_outdoor_siren_v10.zip`（約4.7GB、本体3828本）
- `dataset_outdoor_siren_v10_2_add.zip`（追加705本=学習fold1_room2 675＋幻覚評価fold2_room3 30。zip内パスは本体名に付替済み→順に展開するだけでマージ完了）

- クラス6 = Siren(3型: ピーポー/ウー/消防車) / Horn / BackupBeep(混合レンジ) / BikeBell / CarDrive / Crossing
- 学習 fold1_room1 2400 + fold1_room2 675 / val fold2_room1 600 / test fold3_room1 600（**testは最終1回まで触らない**）
- 評価枠: 交差点 fold2_room9 20・プローブ fold9_room1 48・6シナリオ fold2_room4〜8 100・交通量 fold8_room1 60・幻覚 fold2_room3 30
- 設計の正: `md/design/v10_2_design_2026-07-21.md`。**データを変えたら EXP_NAME を必ず変える**
- 学習は約4〜5時間（T4/100ep）。**切れても再実行すれば last.ckpt から自動再開**（Drive永続化）

---
## 1. GPU 確認

In [ ]:
import torch
assert torch.cuda.is_available(), '⚠ GPUがありません。ランタイム→T4 GPU を選択してください'
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Drive マウントと設定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ==== 設定 ====
DRIVE_DATA = '/content/drive/MyDrive/PSELDNets_data'
DRIVE_LOGS = '/content/drive/MyDrive/PSELDNets_logs'
DRIVE_CKPT = '/content/drive/MyDrive/PSELDNets_ckpts'
DATASET    = 'outdoor_siren_v10'
EXP_NAME   = 'outdoor_siren_v10_2_run1'

import os
for d in [DRIVE_DATA, DRIVE_LOGS, DRIVE_CKPT]:
    os.makedirs(d, exist_ok=True)
MAIN_ZIP = f'{DRIVE_DATA}/dataset_outdoor_siren_v10.zip'
ADD_ZIP  = f'{DRIVE_DATA}/dataset_outdoor_siren_v10_2_add.zip'
assert os.path.exists(MAIN_ZIP), '⚠ 本体zipがDriveにありません'
assert os.path.exists(ADD_ZIP), '⚠ 追補zip(v10_2_add)がDriveにありません'
print(f'OK: main {os.path.getsize(MAIN_ZIP)/1e9:.2f}GB / add {os.path.getsize(ADD_ZIP)/1e6:.0f}MB')

## 3. リポジトリ clone

In [ ]:
import os

REPO = '/content/PSELDNets'

if not os.path.exists(f'{REPO}/src'):
    !git clone https://github.com/Jinbo-Hu/PSELDNets {REPO}
else:
    print(f'既に存在: {REPO}')

os.chdir(REPO)
print(f'CWD: {os.getcwd()}')

## 4. 依存インストール

In [ ]:
!pip install -q \
    librosa \
    soundfile \
    lightning==2.2.1 \
    hydra-core==1.3.2 \
    hydra-colorlog==1.2.0 \
    hydra-joblib-launcher==1.2.0 \
    torchmetrics==1.3.1

import numpy, lightning, torchmetrics, librosa
print(f'numpy {numpy.__version__} / lightning {lightning.__version__} / '
      f'torchmetrics {torchmetrics.__version__} / librosa {librosa.__version__}')

## 5. 事前学習チェックポイント（Drive キャッシュ → なければ HF から）

In [ ]:
import shutil

os.makedirs('ckpts', exist_ok=True)
CKPT = 'ckpts/mACCDOA-HTSAT-0.567.ckpt'
CACHE = f'{DRIVE_CKPT}/mACCDOA-HTSAT-0.567.ckpt'

if not os.path.exists(CKPT):
    if os.path.exists(CACHE):
        print('Drive キャッシュからコピー...')
        shutil.copy(CACHE, CKPT)
    else:
        print('HuggingFace からダウンロード...')
        from huggingface_hub import hf_hub_download
        src = hf_hub_download(repo_id='Jinbo-HU/PSELDNets',
                              filename='model/mACCDOA-HTSAT-0.567.ckpt',
                              repo_type='dataset')
        shutil.copy(src, CKPT)
        shutil.copy(CKPT, CACHE)   # 次回用に Drive にキャッシュ
print(f'OK: {CKPT} ({os.path.getsize(CKPT)/1e6:.0f} MB)')

## 6. データセット展開（本体4.7GB＋追補、10分前後）

追補zipは中のパスが本体名（datasets/outdoor_siren_v10/）に付替済みなので、順に展開するだけでマージされる。

In [ ]:
import zipfile, os

if not os.path.exists(f'datasets/{DATASET}/foa'):
    with zipfile.ZipFile(MAIN_ZIP) as z:
        z.extractall('.')
    print('main unzipped')
else:
    print('本体は展開済み')

# 追補は毎回上書き展開でよい（同一ビット・冪等）
with zipfile.ZipFile(ADD_ZIP) as z:
    z.extractall('.')
print('add unzipped')

n_foa = len(os.listdir(f'datasets/{DATASET}/foa'))
n_meta = len(os.listdir(f'datasets/{DATASET}/metadata'))
n_cls = len(open('datasets/cls_indices_train.tsv').readlines())
print(f'foa: {n_foa} / metadata: {n_meta} / classes: {n_cls}')
assert n_foa == 4533 and n_meta == n_foa and n_cls == 6, '⚠ 中身が想定と違います(3828+705=4533)'

### 6b. 空ラベル＋検品FAILの自動除外（必ず前処理の前に実行）

v9のmix035手動除外の一般化。学習4room（fold1_room1/fold1_room2/fold2_room1/fold3_room1）の空ラベルCSVと検品FAILは3点セット削除、評価専用room（fold2_room3〜9等）の空CSVはダミー行で無害化（testモードはラベルを損失に使わない。採点はローカルのscene.jsonが正）。

In [ ]:
import os
import glob
import shutil

# 検品FAIL（ローカルの inspection.csv で確認済み。v10.2生成の検品で増えたら追記）
INSPECT_FAIL = ['fold2_room1_mix119']

removed_by_split = {}
for room in ("fold1_room1", "fold1_room2", "fold2_room1", "fold3_room1"):
    removed = []
    for f in sorted(glob.glob(f'datasets/{DATASET}/metadata/{room}_*.csv')):
        stem = os.path.basename(f)[:-4]
        if os.path.getsize(f) > 0 and stem not in INSPECT_FAIL:
            continue
        flac = f'datasets/{DATASET}/foa/{stem}.flac'
        mask = f'datasets/{DATASET}/masks/{stem}.csv'
        os.remove(f)
        if os.path.exists(flac):
            os.remove(flac)
        if os.path.exists(mask):
            os.remove(mask)
        removed.append(stem)
    removed_by_split[room] = removed
    print(f'{room}: removed {len(removed)} clips', removed if removed else '')

total = sum(len(v) for v in removed_by_split.values())
print('---')
print(f'total removed: {total}')

# --- 評価専用room（学習4room以外）の0バイトCSVにダミー行を書き込む ---
TRAIN_ROOMS = ("fold1_room1", "fold1_room2", "fold2_room1", "fold3_room1")
patched = []
for f in sorted(glob.glob(f'datasets/{DATASET}/metadata/*.csv')):
    stem = os.path.basename(f)[:-4]
    if stem.startswith(TRAIN_ROOMS):
        continue
    if os.path.getsize(f) == 0:
        with open(f, 'w') as fh:
            fh.write('0,0,0,0,0\n')
        patched.append(stem)
print(f'eval-only rooms: dummy-row patched {len(patched)} clips',
      patched if patched else '')
shutil.rmtree('_hdf5', ignore_errors=True)
print('cleaned _hdf5 -> 次に前処理セルを実行してください')

## 7. 設定ファイル（本体＋推論6種: val / 交差点 / プローブ / 6シナリオ / 交通量 / 幻覚）

roomsフィルタは部分文字列マッチ。学習は [fold1_room1, fold1_room2]（v10.2追加込み）。

In [ ]:
data_yaml = f"""audio_type: foa
audio_feature: logmelIV
sample_rate: 24000
nfft: 1024
n_mels: 64
hoplen: 240
window: hann

train_chunklen_sec: 10
train_hoplen_sec: 10
test_chunklen_sec: 10
test_hoplen_sec: 10

train_dataset:
  {DATASET}: [fold1_room1, fold1_room2]
valid_dataset:
  {DATASET}: [fold2_room1]
test_dataset:
  {DATASET}: [fold3_room1]
"""

exp_yaml = f"""# @package _global_
defaults:
 - override /data: {DATASET}.yaml
 - override /loss: multi_accdoa.yaml
 - _self_

task_name: {DATASET}

model:
  batch_size: 8
  kwargs:
    pretrained_path: ckpts/mACCDOA-HTSAT-0.567.ckpt
    audioset_pretrain: false
  optimizer:
    kwargs: {{lr: 0.0003}}
  lr_scheduler:
    kwargs: {{step_size: 60}}

trainer:
  max_epochs: 100
  check_val_every_n_epoch: 5
"""

open(f'configs/data/{DATASET}.yaml', 'w').write(data_yaml)
open(f'configs/experiment/{DATASET}.yaml', 'w').write(exp_yaml)
for tag, rooms in [('valinfer', '[fold2_room1]'),
                   ('scenario', '[fold2_room9]'),
                   ('probe', '[fold9_room1]'),
                   ('scn2', '[fold2_room4, fold2_room5, fold2_room6, fold2_room7, fold2_room8]'),
                   ('v10a', '[fold8_room1]'),
                   ('halluc', '[fold2_room3]')]:
    d = data_yaml.replace(f'test_dataset:\n  {DATASET}: [fold3_room1]',
                          f'test_dataset:\n  {DATASET}: {rooms}')
    open(f'configs/data/{DATASET}_{tag}.yaml', 'w').write(d)
    e = exp_yaml.replace(f'override /data: {DATASET}.yaml',
                         f'override /data: {DATASET}_{tag}.yaml')
    open(f'configs/experiment/{DATASET}_{tag}.yaml', 'w').write(e)
print('wrote configs (v10 + 6 infer variants)')

## 8. 前処理 → HDF5（初回のみ、20〜30分規模）

In [ ]:
IDX = f'_hdf5/data/24000fs/wav/dev/{DATASET}_10sChunklen_10sHoplen_train.csv'
if not os.path.exists(IDX):
    !python src/preproc.py dataset={DATASET}
else:
    print('前処理済み')
!head -3 {IDX}

## 9. 最終チェック

In [ ]:
checks = [
    ('ckpts/mACCDOA-HTSAT-0.567.ckpt',     '事前学習チェックポイント'),
    ('datasets/cls_indices_train.tsv',      'クラス辞書 TSV (6クラス)'),
    (f'datasets/{DATASET}/foa',             'FOA 音声 (約4500)'),
    (f'datasets/{DATASET}/metadata',        'ラベル CSV'),
    (f'configs/experiment/{DATASET}.yaml',  '実験設定'),
    (IDX,                                   '前処理インデックス'),
]
for path, name in checks:
    ok = os.path.exists(path) and (not os.path.isdir(path) or len(os.listdir(path)) > 0)
    print(f'  [{"OK" if ok else "NG"}] {name}')

## 10. 学習（T4 で約4〜5時間 / 100epoch）

- **切れても再実行すれば last.ckpt から自動再開**（Drive永続化）→ 寝る前にこのセルまで実行
- val は fold2_room1 のみ。test(fold3) はここでは一切使わない
- **データを変えて学習し直すときは必ず EXP_NAME を変えること**

In [ ]:
LAST = f'{DRIVE_LOGS}/{DATASET}/runs/{EXP_NAME}/checkpoints/last.ckpt'
resume = f'ckpt_path={LAST}' if os.path.exists(LAST) else ''
print('resume:', resume or '(new run)')

!python src/train.py experiment={DATASET} \
    experiment_name={EXP_NAME} \
    paths.log_dir={DRIVE_LOGS} \
    {resume}

## 11. 学習曲線（val 抜粋）

In [ ]:
import re

log_path = f'{DRIVE_LOGS}/{DATASET}/runs/{EXP_NAME}/train.log'
lines = [l for l in open(log_path, errors='ignore')
         if 'val/macro' in l or 'train: loss_all' in l]
print(f'--- {log_path} ---')
for l in lines:
    print(re.sub(r'\x1b\[[0-9;]*m', '', l).rstrip())

vals = [l for l in lines if 'val/macro' in l]
if vals:
    print('\n=== 最終 val/macro ===')
    print(re.sub(r'\x1b\[[0-9;]*m', '', vals[-1]).strip())

## 12. 推論（6セット一括: val / 交差点 / プローブ / 6シナリオ / 交通量 / 幻覚）

予測CSVをセットごとに1本へ連結してDriveに保存 → ローカルの解剖・採点（step8系/step12/step15/step16）が読む。

In [ ]:
import glob, os
cands = sorted(glob.glob('/content/drive/MyDrive/PSELDNets_logs*'))
cands += sorted(glob.glob('/content/drive/.shortcut-targets-by-id/*/PSELDNets_logs'))
best_ckpt = None
for c in cands:
    hits = sorted(glob.glob(f'{c}/{DATASET}/runs/{EXP_NAME}/checkpoints/epoch_*.ckpt'))
    if hits:
        DRIVE_LOGS = c
        best_ckpt = hits[-1]
        break
print('best_ckpt =', best_ckpt)
assert best_ckpt, '⚠ ckptが見えません（学習が終わっていますか）'

for tag in ['valinfer', 'scenario', 'probe', 'scn2', 'v10a', 'halluc']:
    short = 'val' if tag == 'valinfer' else tag
    exp = f'infer_{EXP_NAME}_{short}'
    !python src/infer.py experiment={DATASET}_{tag} \
        mode=test \
        ckpt_path="{best_ckpt}" \
        model.kwargs.pretrained_path=null \
        experiment_name={exp} \
        paths.log_dir={DRIVE_LOGS}
    sub = f'{DRIVE_LOGS}/{DATASET}/runs/{exp}/submissions'
    out_lines = []
    for p in sorted(glob.glob(f'{sub}/*.csv')):
        stem = os.path.basename(p)[:-4]
        for line in open(p):
            if line.strip():
                out_lines.append(f'{stem},{line.strip()}')
    out = f'{DRIVE_DATA}/{exp}_all.csv'
    open(out, 'w').write('\n'.join(out_lines))
    print('wrote', out, len(out_lines), 'lines')

## 13. このあと（ローカル側）

1. 6つの `infer_..._all.csv` をローカルへ → 解剖・通知層採点・シナリオ採点・v10a同時検出・幻覚検定(n=50)
2. fold3(test)は全分析が固まった後に**最終1回だけ**（colab/cell_fold3_eval系を参照）
3. batch=1実時間ベンチ（colab/cell_realtime_bench系）・因果推論（colab/cell_causal_infer系）はv10.2用にパス替えで再利用